# Omnilex Legal Retrieval — Hybrid FAISS+BM25 with HyDE

**Pipeline**: Corpus → BM25+FAISS indices → Few-shot bank → HyDE → Hybrid retrieval → ReAct Agent

**Kaggle Setup**:
- Competition data: `/kaggle/input/competitions/llm-agentic-legal-information-retrieval/`
- Model (Add Model): `/kaggle/input/datasets/charan1996/mistral-7b-gguf/`
- GPU T4 x2, Internet ON, Persistence ON

In [ ]:
# === INSTALL ===
!pip install -q rank-bm25 sentence-transformers faiss-cpu
!pip install -q llama-cpp-python --prefer-binary \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


In [ ]:
# === IMPORTS & CONFIG ===
import os, re, gc, pickle, hashlib, time
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import faiss
from tqdm.notebook import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama

# --- Paths ---
KAGGLE = os.path.exists("/kaggle")
if KAGGLE:
    COMP_PATH = Path("/kaggle/input/competitions/llm-agentic-legal-information-retrieval")
    MODEL_PATH = Path("/kaggle/input/datasets/charan1996/mistral-7b-gguf")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/working/indices")
else:
    ROOT = Path(".").resolve().parent
    COMP_PATH = ROOT / "data"
    MODEL_PATH = ROOT / "models"
    OUTPUT_PATH = ROOT / "output"
    INDEX_PATH = ROOT / "data" / "processed"

INDEX_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

LAWS_CSV = COMP_PATH / "laws_de.csv"
COURTS_CSV = COMP_PATH / "court_considerations.csv"
TRAIN_CSV = COMP_PATH / "train.csv"
DATASET_MODE = "test"
QUERY_FILE = COMP_PATH / f"{DATASET_MODE}.csv"

# --- Hyperparams ---
CFG = {
    "n_ctx": 8192, "n_threads": 8, "n_gpu_layers": -1,
    "max_iterations": 3, "max_tokens": 512, "temperature": 0.1,
    "top_k": 40, "rrf_k": 60, "faiss_k": 100, "bm25_k": 100,
    "hyde_max_tokens": 300, "hyde_temperature": 0.3,
    "hyde_few_shot_count": 3, "examples_per_type": 3,
    "max_synthetic_types": 50,
    "embed_batch_size": 1024,
}

# --- Validate ---
for name, p in {"Laws": LAWS_CSV, "Courts": COURTS_CSV, "Queries": QUERY_FILE, "Train": TRAIN_CSV}.items():
    print(f"  {'OK' if p.exists() else 'MISSING':7s} | {name}: {p}")
gguf = list(MODEL_PATH.rglob("*.gguf"))
assert gguf, f"No .gguf in {MODEL_PATH}"
print(f"  OK      | Model: {gguf[0].name}")
print(f"\n  Mode: {DATASET_MODE} | GPU: {KAGGLE}")


In [ ]:
# === LOAD CORPORA (fast: pandas vectorized, no iterrows) ===
t0 = time.time()

def load_corpus(csv_path, max_rows=None):
    """Load CSV to list of dicts using pandas vectorized ops."""
    df = pd.read_csv(csv_path, usecols=["citation", "text"], nrows=max_rows,
                     dtype={"citation": str, "text": str}, na_filter=False)
    docs = df.to_dict("records")
    print(f"  {csv_path.name}: {len(docs):,} docs ({csv_path.stat().st_size/1e6:.0f} MB)")
    return docs

law_docs = load_corpus(LAWS_CSV)
court_docs = load_corpus(COURTS_CSV, max_rows=100_000)
print(f"  Loaded in {time.time()-t0:.1f}s")

In [ ]:
# === BM25 INDICES (build or load cache) ===
t0 = time.time()

class BM25Index:
    __slots__ = ['documents', 'doc_types', 'index', '_tok_corpus']
    
    def __init__(self, documents=None):
        self.documents = documents or []
        self.doc_types = None
        self.index = None
        self._tok_corpus = []
        if documents:
            self._build()
    
    def _build(self):
        self._tok_corpus = [self._tok(d["text"]) for d in self.documents]
        self.index = BM25Okapi(self._tok_corpus)
    
    @staticmethod
    def _tok(text):
        return [t for t in re.split(r"\W+", text.lower()) if t]
    
    def search(self, query, top_k=40):
        tokens = self._tok(query)
        if not tokens or self.index is None:
            return []
        scores = self.index.get_scores(tokens)
        top_idx = scores.argsort()[-top_k:][::-1]
        return [(int(i), float(scores[i])) for i in top_idx if scores[i] > 0]
    
    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump({"docs": self.documents, "tok": self._tok_corpus}, f, protocol=4)
    
    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            d = pickle.load(f)
        obj = cls.__new__(cls)
        obj.documents = d["docs"]
        obj._tok_corpus = d["tok"]
        obj.doc_types = None
        obj.index = BM25Okapi(obj._tok_corpus)
        return obj

def get_or_build(name, docs, path):
    if Path(path).exists():
        idx = BM25Index.load(path)
        print(f"  {name}: loaded {len(idx.documents):,} docs from cache")
        return idx
    idx = BM25Index(docs)
    idx.save(path)
    print(f"  {name}: built + saved ({len(idx.documents):,} docs)")
    return idx

LAWS_BM25_PATH = INDEX_PATH / "laws_bm25.pkl"
COURTS_BM25_PATH = INDEX_PATH / "courts_bm25.pkl"

laws_idx = get_or_build("Laws", law_docs, LAWS_BM25_PATH)
courts_idx = get_or_build("Courts", court_docs, COURTS_BM25_PATH)
del law_docs, court_docs; gc.collect()
print(f"  BM25 ready in {time.time()-t0:.1f}s")

In [ ]:
# === SENTENCE TRANSFORMER + FAISS EMBEDDINGS ===
t0 = time.time()
print("Loading sentence-transformer...")
st_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cuda' if KAGGLE else 'cpu')
st_model.max_seq_length = 512
DIM = st_model.get_sentence_embedding_dimension()
print(f"  Model: {DIM}d on {st_model.device}")

FAISS_LAWS_PATH = INDEX_PATH / "faiss_laws.npy"
FAISS_COURTS_PATH = INDEX_PATH / "faiss_courts.npy"

def embed_corpus(docs, path, batch_size=None):
    """Embed corpus with GPU-accelerated sentence-transformer. Cache as .npy (faster than pickle)."""
    if path.exists():
        emb = np.load(path)
        print(f"  Loaded {path.name}: {emb.shape}")
        return emb
    bs = batch_size or CFG["embed_batch_size"]
    texts = [f"{d['citation']}: {d['text'][:480]}" for d in docs]
    print(f"  Embedding {len(texts):,} docs (batch={bs})...")
    emb = st_model.encode(texts, normalize_embeddings=True, show_progress_bar=True,
                          batch_size=bs, convert_to_numpy=True).astype('float32')
    np.save(path, emb)
    print(f"  Saved {path.name} ({emb.nbytes/1e6:.0f} MB)")
    return emb

law_emb = embed_corpus(laws_idx.documents, FAISS_LAWS_PATH, batch_size=1024)
court_emb = embed_corpus(courts_idx.documents, FAISS_COURTS_PATH, batch_size=512)

# Build FAISS flat-IP indices
faiss_laws = faiss.IndexFlatIP(DIM)
faiss_laws.add(law_emb)
faiss_courts = faiss.IndexFlatIP(DIM)
faiss_courts.add(court_emb)

# Free raw embeddings — FAISS has its own copy
del law_emb, court_emb; gc.collect()
print(f"\n  FAISS ready: Laws={faiss_laws.ntotal:,} Courts={faiss_courts.ntotal:,} in {time.time()-t0:.1f}s")

In [ ]:
# === LOAD LLM ===
model_file = list(MODEL_PATH.rglob("*.gguf"))[0]
n_gpu = CFG["n_gpu_layers"]

# llama-cpp will fall back to CPU layers if CUDA unavailable
try:
    import torch
    if not torch.cuda.is_available():
        n_gpu = 0
except ImportError:
    pass  # llama-cpp handles GPU detection internally

print(f"Loading {model_file.name} (GPU layers: {n_gpu})...")
llm = Llama(model_path=str(model_file), n_ctx=CFG["n_ctx"],
            n_threads=CFG["n_threads"], n_gpu_layers=n_gpu, verbose=False)
gc.collect()
print("  LLM ready")

In [ ]:
# === TYPE CLASSIFICATION ===
def get_law_type(cit):
    m = re.search(r'\b([A-Z]{2,}[a-z]?)\s*$', cit.strip())
    if m: return m.group(1)
    ms = re.findall(r'\b([A-Z]{2,})\b', cit)
    return ms[-1] if ms else "OTHER"

def get_court_type(cit):
    m = re.match(r'BGE\s+\d+\s+([IVX]+)', cit)
    if m: return f"BGE_{m.group(1)}"
    m = re.match(r'(\d+[A-Z]+)', cit)
    if m: return f"CASE_{m.group(1)}"
    return "OTHER"

laws_idx.doc_types = np.array([get_law_type(d["citation"]) for d in laws_idx.documents])
courts_idx.doc_types = np.array([get_court_type(d["citation"]) for d in courts_idx.documents])

law_type_counts = Counter(laws_idx.doc_types)
court_type_counts = Counter(courts_idx.doc_types)
LAW_TYPES_STR = ", ".join(f"{t}({c})" for t, c in sorted(law_type_counts.items(), key=lambda x: -x[1])[:40])
COURT_TYPES_STR = ", ".join(f"{t}({c})" for t, c in sorted(court_type_counts.items(), key=lambda x: -x[1]))
print(f"Types: {len(law_type_counts)} law, {len(court_type_counts)} court")

In [ ]:
# === FEW-SHOT EXAMPLE BANK (with FAISS index for selection) ===
FEW_SHOT_PATH = INDEX_PATH / "few_shot.pkl"
EPT = CFG["examples_per_type"]

if FEW_SHOT_PATH.exists():
    with open(FEW_SHOT_PATH, "rb") as f:
        _fs = pickle.load(f)
    law_fs_bank, court_fs_bank = _fs["law"], _fs["court"]
    fs_examples, fs_embeddings = _fs["examples"], _fs["embeddings"]
    print(f"  Few-shot loaded: {len(fs_examples)} examples")
else:
    print("Building few-shot bank...")
    # Get real examples from train.csv
    train_df = pd.read_csv(TRAIN_CSV)
    law_cit2text = {d["citation"]: d["text"] for d in laws_idx.documents if d["text"]}
    court_cit2text = {d["citation"]: d["text"] for d in courts_idx.documents if d["text"]}
    
    law_fs_bank = defaultdict(list)
    court_fs_bank = defaultdict(list)
    
    for _, row in train_df.iterrows():
        query = str(row["query"])
        gold = str(row.get("gold_citations", ""))
        if not gold or gold == "nan":
            continue
        for cit in (c.strip() for c in gold.split(";") if c.strip()):
            if cit in law_cit2text:
                t = get_law_type(cit)
                if len(law_fs_bank[t]) < EPT:
                    law_fs_bank[t].append({"query": query[:500], "citation": cit, "text": law_cit2text[cit][:600]})
            elif cit in court_cit2text:
                t = get_court_type(cit)
                if len(court_fs_bank[t]) < EPT:
                    court_fs_bank[t].append({"query": query[:500], "citation": cit, "text": court_cit2text[cit][:600]})
    
    # Synthetic fill for types without real examples (batch LLM)
    def _synth_query(text, dtype="law"):
        task = "Schweizer Gesetzesartikel" if dtype == "law" else "Schweizer Gerichtserwägung"
        p = (f"[INST] Gegeben der folgende {task}, schreibe eine kurze rechtliche Frage auf Deutsch. "
             f"Nur die Frage.\n\nText: {text[:400]}\n\nFrage: [/INST]")
        try:
            return llm(p, max_tokens=80, temperature=0.3, stop=["[INST]", "</s>", "\n\n"])["choices"][0]["text"].strip()
        except Exception:
            return f"Frage zu {task}?"
    
    # Fill gaps from corpus samples (types with < EPT real examples)
    corpus_law_samples = defaultdict(list)
    for d in laws_idx.documents:
        t = get_law_type(d["citation"])
        if len(corpus_law_samples[t]) < 5 and len(law_fs_bank.get(t, [])) < EPT:
            # Skip citations already in bank
            if d["citation"] not in {e["citation"] for e in law_fs_bank.get(t, [])}:
                corpus_law_samples[t].append(d)
    corpus_court_samples = defaultdict(list)
    for d in courts_idx.documents:
        t = get_court_type(d["citation"])
        if len(corpus_court_samples[t]) < 5 and len(court_fs_bank.get(t, [])) < EPT:
            if d["citation"] not in {e["citation"] for e in court_fs_bank.get(t, [])}:
                corpus_court_samples[t].append(d)
    
    # Generate synthetic (capped at max_synthetic_types)
    types_to_fill = list(corpus_law_samples.items())[:CFG["max_synthetic_types"]]
    print(f"  Synthetic: {len(types_to_fill)} law types...")
    for t, samples in tqdm(types_to_fill, desc="Synth-law"):
        needed = EPT - len(law_fs_bank.get(t, []))
        for d in samples[:needed]:
            q = _synth_query(d["text"], "law")
            law_fs_bank[t].append({"query": q, "citation": d["citation"], "text": d["text"][:600]})
    
    types_to_fill = list(corpus_court_samples.items())[:CFG["max_synthetic_types"]]
    print(f"  Synthetic: {len(types_to_fill)} court types...")
    for t, samples in tqdm(types_to_fill, desc="Synth-court"):
        needed = EPT - len(court_fs_bank.get(t, []))
        for d in samples[:needed]:
            q = _synth_query(d["text"], "court")
            court_fs_bank[t].append({"query": q, "citation": d["citation"], "text": d["text"][:600]})
    
    # Build flat example list + embed queries for FAISS selection
    fs_examples = []
    fs_queries = []
    for t, exs in law_fs_bank.items():
        for ex in exs:
            fs_examples.append(("law", t, ex))
            fs_queries.append(ex["query"])
    for t, exs in court_fs_bank.items():
        for ex in exs:
            fs_examples.append(("court", t, ex))
            fs_queries.append(ex["query"])
    
    print(f"  Embedding {len(fs_queries)} few-shot queries...")
    fs_embeddings = st_model.encode(fs_queries, normalize_embeddings=True,
                                    batch_size=256, show_progress_bar=True).astype('float32')
    
    # Save
    with open(FEW_SHOT_PATH, "wb") as f:
        pickle.dump({"law": dict(law_fs_bank), "court": dict(court_fs_bank),
                     "examples": fs_examples, "embeddings": fs_embeddings}, f, protocol=4)
    print(f"  Saved: {len(fs_examples)} examples")
    del train_df, law_cit2text, court_cit2text; gc.collect()

# Build few-shot FAISS indices
_law_mask = np.array([i for i, (dt, _, _) in enumerate(fs_examples) if dt == "law"])
_court_mask = np.array([i for i, (dt, _, _) in enumerate(fs_examples) if dt == "court"])

faiss_fs_law = faiss.IndexFlatIP(DIM)
if len(_law_mask): faiss_fs_law.add(fs_embeddings[_law_mask])
faiss_fs_court = faiss.IndexFlatIP(DIM)
if len(_court_mask): faiss_fs_court.add(fs_embeddings[_court_mask])
print(f"  Few-shot FAISS: law={faiss_fs_law.ntotal}, court={faiss_fs_court.ntotal}")

In [ ]:
# === SEARCH FUNCTIONS (HyDE + Hybrid FAISS/BM25 + RRF) ===

_hyde_cache = {}

def _llm_call(prompt, max_tokens=300, temp=0.3, stop=None):
    """Single LLM call with error handling."""
    try:
        return llm(prompt, max_tokens=max_tokens, temperature=temp,
                   stop=stop or ["[INST]", "</s>"])["choices"][0]["text"].strip()
    except Exception:
        return ""

def select_few_shot(query, doc_type="law", n=3):
    """FAISS nearest-neighbor few-shot selection."""
    idx = faiss_fs_law if doc_type == "law" else faiss_fs_court
    mask = _law_mask if doc_type == "law" else _court_mask
    if idx.ntotal == 0:
        return []
    q_vec = st_model.encode([query], normalize_embeddings=True).astype('float32')
    k = min(n, idx.ntotal)
    _, local_ids = idx.search(q_vec, k)
    return [fs_examples[mask[i]][2] for i in local_ids[0] if i >= 0]

def generate_hyde(query, doc_type="law", examples=None):
    """Generate hypothetical German legal document for BM25."""
    key = hashlib.md5(f"{query}:{doc_type}".encode()).hexdigest()
    if key in _hyde_cache:
        return _hyde_cache[key]
    
    if doc_type == "law":
        inst = ("Du bist ein Schweizer Rechtsexperte. Schreibe einen hypothetischen Gesetzesartikel "
                "auf Deutsch der diese Frage beantwortet (~300 Zeichen).")
    else:
        inst = ("Du bist ein Schweizer Rechtsexperte. Schreibe eine hypothetische BGE-Erwägung "
                "auf Deutsch die diese Frage behandelt (~400 Zeichen).")
    
    ex_text = ""
    if examples:
        for ex in examples[:3]:
            ex_text += f"\nFrage: {ex['query'][:250]}\nText: {ex['text'][:300]}\n"
    
    prompt = f"[INST] {inst}{ex_text}\n\nFrage: {query}\n\nText: [/INST]"
    result = _llm_call(prompt, max_tokens=CFG["hyde_max_tokens"], temp=CFG["hyde_temperature"])
    _hyde_cache[key] = result or query
    return _hyde_cache[key]

def hybrid_search(query, doc_type="law", top_k=40):
    """FAISS (English query) + BM25 (German HyDE) → RRF fusion."""
    if doc_type == "law":
        f_idx, b_idx = faiss_laws, laws_idx
    else:
        f_idx, b_idx = faiss_courts, courts_idx
    
    # Few-shot → HyDE
    examples = select_few_shot(query, doc_type, n=CFG["hyde_few_shot_count"])
    hyde_doc = generate_hyde(query, doc_type, examples)
    
    # FAISS semantic search (English query → multilingual space)
    q_vec = st_model.encode([query], normalize_embeddings=True).astype('float32')
    f_scores, f_ids = f_idx.search(q_vec, CFG["faiss_k"])
    faiss_ranks = {int(idx): rank+1 for rank, (s, idx) in enumerate(zip(f_scores[0], f_ids[0])) if idx >= 0 and s > 0}
    
    # BM25 keyword search (German HyDE document)
    bm25_hits = b_idx.search(hyde_doc, CFG["bm25_k"])
    bm25_ranks = {idx: rank+1 for rank, (idx, _) in enumerate(bm25_hits)}
    
    # RRF fusion
    rrf_k = CFG["rrf_k"]
    candidates = set(faiss_ranks) | set(bm25_ranks)
    rrf = {idx: 1/(rrf_k + faiss_ranks.get(idx, CFG["faiss_k"]+1)) + 
               1/(rrf_k + bm25_ranks.get(idx, CFG["bm25_k"]+1))
           for idx in candidates}
    
    ranked = sorted(rrf.items(), key=lambda x: -x[1])[:top_k]
    docs = b_idx.documents
    return [{"citation": docs[i]["citation"], "text": docs[i]["text"][:400],
             "_score": s, "_type": str(b_idx.doc_types[i])} for i, s in ranked]

print("Search functions ready")

In [ ]:
# === REACT AGENT (tools + system prompt with few-shot examples) ===

# -----------------------------------------------------------
# TOOLS — thin wrappers around hybrid_search, returning a formatted
# string for the LLM to read in the Observation step.
# -----------------------------------------------------------
def search_laws(query: str, top_k: int = 20) -> tuple[str, list[str]]:
    """Search Swiss federal laws (SR). Returns (formatted_obs, citation_list)."""
    results = hybrid_search(query, doc_type="law", top_k=top_k)
    obs = "\n".join(f"- {r['citation']}: {r['text'][:200]}" for r in results) or "No results."
    return obs[:1200], [r["citation"] for r in results]

def search_courts(query: str, top_k: int = 20) -> tuple[str, list[str]]:
    """Search Swiss Federal Court decisions (BGE). Returns (formatted_obs, citation_list)."""
    results = hybrid_search(query, doc_type="court", top_k=top_k)
    obs = "\n".join(f"- {r['citation']}: {r['text'][:200]}" for r in results) or "No results."
    return obs[:1200], [r["citation"] for r in results]

TOOLS = {"search_laws": search_laws, "search_courts": search_courts}

# -----------------------------------------------------------
# SYSTEM PROMPT — tool docs + ReAct format + few-shot examples
# -----------------------------------------------------------
SYSTEM_PROMPT = f"""You are a Swiss legal research assistant with access to two search tools:

TOOLS:
1. search_laws(query)
   - Searches Swiss federal laws (Systematische Rechtssammlung / SR)
   - Returns: list of law citations (e.g. "Art. 1 OR", "Art. 117 StGB") with German excerpts
   - Use for: statutory provisions, legal definitions, code articles
   - Available law types: {LAW_TYPES_STR}

2. search_courts(query)
   - Searches Swiss Federal Court decisions (Bundesgerichtsentscheide / BGE)
   - Returns: list of court citations (e.g. "BGE 127 III 248 E. 3.1") with German excerpts
   - Use for: case law, judicial interpretation, precedent
   - Available court types: {COURT_TYPES_STR}

REASONING FORMAT (REQUIRED — repeat for each step):

Thought: [your reasoning about what to search for next]
Action: [search_laws OR search_courts]
Action Input: [English search query with specific legal terms]

After you see "Observation:" with the search results, continue with another Thought/Action,
OR conclude with:

Final Answer: [comma-separated list of the most relevant citations from your searches]

RULES:
- ALWAYS perform at least one search_laws AND one search_courts call before Final Answer
- Use 2-3 search iterations max; vary your query terms each time
- Keep Action Input in English; the tools handle cross-lingual matching
- Final Answer must list ONLY citations exactly as returned in Observations

---
Example 1 — Contract law:
Query: What are the requirements for forming a valid contract under Swiss law?

Thought: I need to find the statutory requirements for contract formation in Swiss obligations law.
Action: search_laws
Action Input: requirements valid contract formation obligations consent

Observation: - Art. 1 OR: Zum Abschluss eines Vertrages ist die übereinstimmende gegenseitige Willensäusserung der Parteien erforderlich...
- Art. 11 OR: Verträge bedürfen zu ihrer Gültigkeit nur dann einer besonderen Form...

Thought: Now I need Federal Court case law on contract validity and consent.
Action: search_courts
Action Input: contract validity mutual consent formation requirements

Observation: - BGE 127 III 248 E. 3.1: Die Voraussetzungen eines gültigen Vertragsschlusses...
- BGE 132 III 268 E. 2.3.1: Übereinstimmende Willenserklärungen...

Final Answer: Art. 1 OR, Art. 11 OR, BGE 127 III 248 E. 3.1, BGE 132 III 268 E. 2.3.1
---
Example 2 — Criminal law:
Query: Under what conditions can a person be held criminally liable for negligent homicide?

Thought: I should search the criminal code for negligent homicide provisions.
Action: search_laws
Action Input: negligent homicide criminal liability fahrlaessige Toetung

Observation: - Art. 117 StGB: Wer fahrlässig den Tod eines Menschen verursacht...
- Art. 12 StGB: Wer ein Verbrechen oder Vergehen fahrlässig begeht...

Thought: I need court precedent on the duty of care standard.
Action: search_courts
Action Input: negligent homicide duty of care Sorgfaltspflicht standard

Observation: - BGE 135 IV 56 E. 2.1: Die Sorgfaltspflicht beurteilt sich nach...
- BGE 145 IV 154 E. 2: Pflichtwidrige Unvorsichtigkeit setzt voraus...

Final Answer: Art. 117 StGB, Art. 12 StGB, BGE 135 IV 56 E. 2.1, BGE 145 IV 154 E. 2
---

Now answer the user's query below. Begin with "Thought:".
"""

# -----------------------------------------------------------
# AGENT LOOP
# -----------------------------------------------------------
_ACTION_RE = re.compile(
    r"Action:\s*(search_\w+)\s*\nAction Input:\s*(.+?)(?=\n(?:Thought|Action|Final|Observation)|$)",
    re.I | re.S,
)

def _parse_actions(text: str) -> list[tuple[str, str]]:
    """Extract all (tool_name, query) pairs from an LLM response."""
    actions = [(m.group(1).strip().lower(), m.group(2).strip()) for m in _ACTION_RE.finditer(text)]
    if not actions:
        am = re.search(r"Action:\s*(search_\w+)", text, re.I)
        ai = re.search(r"Action Input:\s*(.+?)$", text, re.I | re.M)
        if am and ai:
            actions.append((am.group(1).strip().lower(), ai.group(1).strip()))
    return actions

def _extract_cits(text: str) -> list[str]:
    """Regex-extract citations from agent free text (used as fallback)."""
    cits = []
    cits += re.findall(r"Art\.?\s*\d+[a-z]?\s+(?:Abs\.?\s*\d+\s+)?[A-Z]{2,}", text)
    cits += re.findall(r"BGE\s+\d{1,3}\s+[IVX]+[a-z]?\s+\d+(?:\s+E\.\s*\d+)?", text)
    cits += re.findall(r"SR\s*\d{3}(?:\.\d+)*(?:\s+Art\.?\s*\d+[a-z]?)?", text)
    return list(dict.fromkeys(cits))

def run_agent(query: str, verbose: bool = False) -> list[str]:
    """ReAct loop: alternating LLM Thought/Action and tool Observation.
    Returns deduplicated list of citations collected from all tool calls + final answer."""
    all_citations: list[str] = []
    conv = f"[INST] {SYSTEM_PROMPT}\n\nQuery: {query}\n\nThought: [/INST]"

    for it in range(CFG["max_iterations"]):
        # Trim conversation if it grows too large
        if len(conv) > 28000:
            inst_end = conv.find("[/INST]")
            conv = conv[: inst_end + 7] + "\n...[truncated]...\n" + conv[-12000:]

        resp = _llm_call(
            conv,
            max_tokens=CFG["max_tokens"],
            temp=CFG["temperature"],
            stop=["Observation:", "[INST]", "</s>"],
        )
        if not resp:
            break

        if verbose:
            print(f"  [Iter {it+1}] {resp[:250]}")

        # Final Answer terminates the loop
        if "Final Answer:" in resp:
            all_citations += _extract_cits(resp.split("Final Answer:", 1)[1])
            break

        actions = _parse_actions(resp)
        if not actions:
            all_citations += _extract_cits(resp)
            break

        # Execute every parsed tool call
        obs_parts = []
        for tool_name, tool_query in actions:
            tool = TOOLS.get(tool_name)
            if tool is None:
                obs_parts.append(f"Error: unknown tool '{tool_name}'")
                continue
            obs, cits = tool(tool_query)
            all_citations += cits
            obs_parts.append(obs)
            if verbose:
                print(f"    [{tool_name}] → {len(cits)} citations")

        conv += (
            resp
            + "\nObservation: " + "\n".join(obs_parts)
            + "\n\n[INST] Continue with another Thought/Action, or give the Final Answer. [/INST]\n\nThought:"
        )

    return list(dict.fromkeys(all_citations))  # dedupe, preserve order

print("Agent ready")
print(f"  System prompt: {len(SYSTEM_PROMPT):,} chars")
print(f"  Tools: {list(TOOLS.keys())}")


In [ ]:
# === QUICK TEST ===
test_q = "What are the requirements for a valid contract under Swiss law?"
cits = run_agent(test_q, verbose=True)
print(f"\n→ {len(cits)} citations: {cits[:10]}")

In [ ]:
# === GENERATE PREDICTIONS ===
test_df = pd.read_csv(QUERY_FILE)
print(f"Processing {len(test_df)} queries...")

predictions = []
t0 = time.time()

for idx, (_, row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df), desc="Queries"), 1):
    citations = run_agent(str(row["query"]))
    predictions.append({"query_id": row["query_id"], "predicted_citations": ";".join(citations)})
    
    # Checkpoint every 10
    if idx % 10 == 0:
        elapsed = time.time() - t0
        rate = idx / elapsed * 60
        remaining = len(test_df) - idx
        print(f"  [{idx}/{len(test_df)}] {rate:.1f} queries/min, ~{remaining/rate:.0f} min remaining")
        pd.DataFrame(predictions).to_csv(OUTPUT_PATH / "submission_checkpoint.csv", index=False)

predictions_df = pd.DataFrame(predictions)
avg_cits = predictions_df['predicted_citations'].apply(lambda x: len(x.split(';')) if x else 0).mean()
print(f"\nDone in {(time.time()-t0)/60:.1f} min | Avg citations: {avg_cits:.1f}")

In [ ]:
# === SAVE SUBMISSION ===
sub_path = OUTPUT_PATH / "submission.csv"
predictions_df.to_csv(sub_path, index=False)
print(f"Saved: {sub_path} ({sub_path.stat().st_size/1024:.1f} KB)")
predictions_df.head()

In [ ]:
# === EVALUATION (val mode only) ===
if DATASET_MODE == "val" and "gold_citations" in test_df.columns:
    f1s = []
    for i, row in test_df.iterrows():
        pred = set(c.strip() for c in str(predictions_df.iloc[i]["predicted_citations"]).split(";") if c.strip())
        gold = set(c.strip() for c in str(row["gold_citations"]).split(";") if c.strip())
        if not pred and not gold: f1s.append(1.0); continue
        if not pred or not gold: f1s.append(0.0); continue
        tp = len(pred & gold)
        p, r = tp/len(pred), tp/len(gold)
        f1s.append(2*p*r/(p+r) if (p+r) > 0 else 0.0)
    print(f"MACRO F1: {np.mean(f1s):.4f}")
else:
    print("Submit to competition for score.")